In [ ]:
# Notebook implementation for learning TD Learning

In [ ]:
import gymnasium as gym
import math
import numpy as np

In [ ]:
env = gym.make("FrozenLake-v1")
#env = gym.make('FrozenLake-v1', desc=None, map_name="4x4", is_slippery=True)
#env = gym.make('FrozenLake-v1', desc=None, map_name="8x8", is_slippery=True)

In [ ]:
# Sample an MDP using TD learning
# n is look ahead value, for example TD(0) has n = 1
def sample(start_obs, n):
    #obs, info = env.reset()
    episode = []
    obs = start_obs
    for i in range(n):
        action = env.action_space.sample()
        nextObs, reward, terminated, trunc, info = env.step(action)
        episode.append((obs, action, reward, nextObs))
        obs = nextObs
        if trunc or terminated:
            break
    return episode, trunc or terminated

In [ ]:
obs, info = env.reset()

ended = False
while not ended:
    episode, ended = sample(obs, 1)
    print(episode)

In [ ]:
obs, info = env.reset()

values = np.zeros((env.observation_space.n))
#visit_count = np.zeros((env.observation_space.n))

lamb = 0.9 # Discount factor lambda

# Run TD learning for some number of iterations
n = 1 # n = 1 is TD(0)
iter = 10000
for i in range(iter):
    # Sample the state    
    episode, ended = sample(obs, n)
    
    #G_t_1 = 0 # Intialise return from after the final step to 0
    if len(episode) < 1:
        obs, info = env.reset()
        continue
    
    G_t_1 = values[episode[-1][-1]]

    # Go through each step in the episode in reverse order
    for step in reversed(episode):
        
        o, a, r, n_o = step
        
        # G_t = r_t + lamda * G_t+1
        G_t = r + lamb * G_t_1

        # Update the values with the value of the return from this state
        values[n_o] += G_t
        # Update the visit count so we can average to get the state values
        #visit_count[n_o] += 1

        # Update G_(t+1) for the next iteration
        G_t_1 = G_t
    
    obs = nextObs
    
    if ended:
        obs, info = env.reset()
        #print(ended, " ", i)

In [ ]:
width = int(math.sqrt(env.observation_space.n))

def print_board(board):
    step = int(math.sqrt(env.observation_space.n))
    for i in np.arange(int(env.observation_space.n), step=step):
        for j in np.arange(step):
            print(f"{board[i+j]:.5f}", end=', ')
        print()

In [ ]:
# Print the accumulated value of each state (we need to average of the number of visits for the true MC estimates)
print_board(values)